# Notebook 02b — Génération Q&R ASYNCHRONE (3 datasets)

Version optimisée avec `anthropic.AsyncAnthropic` : 10 requêtes en parallèle, ~3 min estimées.
Meme logique que 02_dataset_builder.ipynb mais avec asyncio pour la vitesse.

**Outputs** : `train.json` (300 paires) + `test.json` (120 paires)

## 0. Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/llm-integration-study/'

## 1. Installation

In [ ]:
# groq pour l'API async, nest_asyncio pour utiliser await dans Colab, json-repair pour le JSON malformé
!pip install -q groq nest_asyncio json-repair

## 2. Imports, configuration et clé API

In [ ]:
import os, json, time, asyncio, random, getpass, re
import nest_asyncio
from groq import AsyncGroq
from tqdm.notebook import tqdm

# Indispensable pour utiliser 'await' dans les cellules Colab
nest_asyncio.apply()

RAW_PATH       = os.path.join(BASE_PATH, 'data', 'raw')
PROCESSED_PATH = os.path.join(BASE_PATH, 'data', 'processed')
os.makedirs(PROCESSED_PATH, exist_ok=True)

GROQ_MODEL = 'llama-3.1-8b-instant'
BATCH_SIZE = 20   # Groq Developer supporte la concurrence
THROTTLE_S = 0.2  # pause légère entre batches

print(f'Modèle      : {MODEL}')
print(f'Batch size  : {BATCH_SIZE} requêtes concurrentes')
print(f'nest_asyncio activé.')

In [ ]:
# Saisie sécurisée de la clé — https://console.groq.com → API Keys
api_key      = getpass.getpass('Entre ta clé Groq API : ')
groq_client  = AsyncGroq(api_key=api_key)
print('Client AsyncGroq initialisé (plan Developer).')

## 3. Chargement des données brutes

In [ ]:
def load_json(path, label=''):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"  Chargé ({label}) : {len(data)} docs")
        return data
    except FileNotFoundError:
        print(f"  [ERROR] Introuvable : {path}")
        return []
    except json.JSONDecodeError as e:
        print(f"  [ERROR] JSON invalide : {e}")
        return []

print("Chargement des 3 datasets bruts...")
wiki_docs    = load_json(os.path.join(RAW_PATH, 'wikipedia_technique.json'), 'Wikipedia technique')
arxiv_docs   = load_json(os.path.join(RAW_PATH, 'arxiv.json'),               'Arxiv multisauts')
lemonde_docs = load_json(os.path.join(RAW_PATH, 'lemonde.json'),             'Le Monde temporel')
print(f"\nTotal : {len(wiki_docs)+len(arxiv_docs)+len(lemonde_docs)} documents")

## 4. Génération async des paires Q&R

In [ ]:
# ── Prompt standard (Wikipedia + Le Monde) ───────────────────────────────────
STANDARD_PROMPT = """Tu es un expert. À partir du texte ci-dessous, génère EXACTEMENT 5 paires question-réponse EN FRANÇAIS.

Génère dans cet ordre :
1. Question FACTUELLE (fait précis, date, chiffre, entité nommée)
2. Question FACTUELLE (idem)
3. Question de SYNTHÈSE (compare ou résume plusieurs concepts)
4. Question de SYNTHÈSE (idem)
5. Question de COMPRÉHENSION (causalité, implication, pourquoi/comment)

RÈGLE ABSOLUE pour "context" : copie MOT POUR MOT un extrait du texte (20-150 mots) qui contient/justifie la réponse.
INTERDIT d'écrire "Texte source", "référence au texte" ou le titre seul.

Réponds UNIQUEMENT avec le tableau JSON.
Chaque objet : {{"question":"...","answer":"...","context":"...","type":"..."}}
Types autorisés : "factuel", "synthese", "comprehension"

Exemple :
[{{"question":"En quelle année X a été fondé ?","answer":"2020","context":"X a été fondé en 2020 par des chercheurs issus de Google Brain, avec pour objectif de...","type":"factuel"}}]

Texte source :
{content}"""

# ── Prompt Arxiv simples (3 questions par papier) ────────────────────────────
ARXIV_SIMPLE_PROMPT = """Tu es un expert en IA. À partir de ce résumé de papier scientifique, génère EXACTEMENT 3 questions factuelles simples EN FRANÇAIS.

Chaque question porte sur UN SEUL fait (méthode utilisée, métrique obtenue, dataset employé, contribution principale).
Chaque réponse tient en 1-2 phrases courtes.

RÈGLE ABSOLUE pour "context" : copie MOT POUR MOT un extrait du résumé (20-100 mots) qui contient la réponse.
INTERDIT d'écrire "Texte source" ou similaire.

Réponds UNIQUEMENT avec le tableau JSON.
Chaque objet : {{"question":"...","answer":"...","context":"...","type":"simple"}}

Exemple :
[{{"question":"Quel dataset est utilisé pour l'évaluation ?","answer":"MMLU","context":"We evaluate our model on the MMLU benchmark, achieving state-of-the-art performance across 57 tasks.","type":"simple"}}]

Résumé :
{content}"""

# ── Prompt Arxiv complexes / multi-sauts (2 questions par papier) ─────────────
ARXIV_COMPLEX_PROMPT = """Tu es un expert en IA. À partir de ce résumé de papier scientifique, génère EXACTEMENT 2 questions complexes EN FRANÇAIS.

Ces questions nécessitent de RELIER PLUSIEURS INFORMATIONS du texte pour répondre (multi-sauts de raisonnement).
Par exemple : "Pourquoi la méthode X obtient-elle de meilleurs résultats que Y sur Z ?"
Chaque réponse fait 3-4 phrases et synthétise plusieurs éléments du résumé.

RÈGLE ABSOLUE pour "context" : copie MOT POUR MOT 1-2 extraits du résumé (30-200 mots) qui, ensemble, permettent de répondre.

Réponds UNIQUEMENT avec le tableau JSON.
Chaque objet : {{"question":"...","answer":"...","context":"...","type":"complexe"}}

Résumé :
{content}"""

GROQ_MODEL  = "llama-3.1-8b-instant"
THROTTLE_S  = 0.5    # Groq Developer — pas de rate limit pratique
MAX_CONTENT = 3000   # chars envoyés au LLM

In [ ]:
VALID_TYPES = {"factuel", "synthese", "comprehension", "simple", "complexe"}

async def _call_groq_async(prompt, retries=4):
    for attempt in range(retries):
        try:
            resp = await groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}]
            )
            return resp.choices[0].message.content
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate_limit' in err.lower():
                wait = 5 * (2 ** attempt)  # 5s, 10s, 20s
                print(f"    [QUOTA] Attente {wait}s...")
                await asyncio.sleep(wait)
            else:
                print(f"    [ERROR] {err[:80]}")
                await asyncio.sleep(1)
    return ""

def _parse_pairs(raw_text, document, forced_type=None):
    content = document.get('content', '')
    json_str = _extract_and_repair(raw_text)
    pairs = json.loads(json_str)
    if not isinstance(pairs, list):
        return []
    validated = []
    for pair in pairs:
        if not (isinstance(pair, dict) and pair.get('question') and pair.get('answer')):
            continue
        q_type = forced_type or str(pair.get('type', 'factuel')).lower().strip()
        if q_type not in VALID_TYPES:
            q_type = forced_type or 'factuel'
        ctx = str(pair.get('context', '')).strip()
        if len(ctx) < 30:
            ctx = content[:400]
        validated.append({
            "question":      str(pair['question']).strip(),
            "answer":        str(pair['answer']).strip(),
            "context":       ctx,
            "source_id":     document.get('id', ''),
            "source":        document.get('source', ''),
            "langue":        "fr",
            "title":         document.get('title', ''),
            "date":          document.get('date', ''),
            "dataset_type":  document.get('dataset_type', ''),
            "question_type": q_type,
        })
    return validated

async def generate_standard_qa_async(document, retries=4):
    content = document.get('content', '')
    if len(content) < 100:
        return []
    prompt = STANDARD_PROMPT.format(content=content[:MAX_CONTENT])
    for attempt in range(retries):
        try:
            raw = await _call_groq_async(prompt)
            if not raw:
                continue
            pairs = _parse_pairs(raw, document)
            if pairs:
                return pairs
        except (json.JSONDecodeError, ValueError):
            await asyncio.sleep(2 ** attempt)
    return []

async def generate_arxiv_qa_async(document, retries=4):
    content = document.get('content', '')
    if len(content) < 100:
        return []
    all_pairs = []
    for prompt_template, forced_type, n_expected in [
        (ARXIV_SIMPLE_PROMPT,  'simple',   3),
        (ARXIV_COMPLEX_PROMPT, 'complexe', 2),
    ]:
        prompt = prompt_template.format(content=content[:MAX_CONTENT])
        for attempt in range(retries):
            try:
                raw = await _call_groq_async(prompt)
                if not raw:
                    break
                pairs = _parse_pairs(raw, document, forced_type=forced_type)
                if pairs:
                    all_pairs.extend(pairs[:n_expected])
                    break
            except (json.JSONDecodeError, ValueError):
                await asyncio.sleep(2 ** attempt)
    return all_pairs

print("Fonctions async chargees.")

In [ ]:
async def generate_all_qa(docs_wiki, docs_arxiv, docs_lemonde):
    """Génère toutes les Q&R en parallèle par batches."""
    sem = asyncio.Semaphore(BATCH_SIZE)

    async def process_doc(doc):
        async with sem:
            dtype = doc.get('dataset_type', '')
            if dtype == 'multisauts':
                pairs = await generate_arxiv_qa_async(doc)
            else:
                pairs = await generate_standard_qa_async(doc)
            return pairs

    all_docs = docs_wiki + docs_arxiv + docs_lemonde
    tasks = [process_doc(d) for d in all_docs]
    results = []
    failed  = 0
    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Q&R async"):
        try:
            pairs = await coro
            results.extend(pairs)
        except Exception as e:
            print(f"  [ERROR] {e}")
            failed += 1
        await asyncio.sleep(THROTTLE_S)
    print(f"\nGeneres : {len(results)} paires  |  echecs : {failed}")
    return results

# Run
nest_asyncio.apply()
all_qa_pairs = asyncio.run(generate_all_qa(wiki_docs, arxiv_docs, lemonde_docs))

## 5. Mélange et division train / test

In [ ]:
import random
random.seed(42)

def exact_split(pairs, n_train, n_test, label=''):
    """Sélectionne exactement n_train + n_test paires (shuffle reproductible)."""    shuffled = pairs[:]
    random.shuffle(shuffled)
    train = shuffled[:n_train]
    test  = shuffled[n_train:n_train + n_test]
    total = len(pairs)
    if total < n_train + n_test:
        print(f"  [WARN] {label}: seulement {total} paires dispo pour {n_train}+{n_test} demandées")
    return train, test

# ── Wikipedia : 100 train + 40 test ─────────────────────────────────────────
wiki_train, wiki_test = exact_split(wiki_qa, 100, 40, 'Wikipedia')

# ── Arxiv : équilibre simple/complexe ────────────────────────────────────────
arxiv_s_train, arxiv_s_test = exact_split(arxiv_simple,  50, 20, 'Arxiv simple')
arxiv_c_train, arxiv_c_test = exact_split(arxiv_complex, 50, 20, 'Arxiv complexe')
arxiv_train = arxiv_s_train + arxiv_c_train
arxiv_test  = arxiv_s_test  + arxiv_c_test

# ── Le Monde : 100 train + 40 test ───────────────────────────────────────────
lemonde_train, lemonde_test = exact_split(lemonde_qa, 100, 40, 'Le Monde')

# ── Assemblage final ─────────────────────────────────────────────────────────
train_data = wiki_train + arxiv_train + lemonde_train
test_data  = wiki_test  + arxiv_test  + lemonde_test
random.shuffle(train_data)
random.shuffle(test_data)

# Re-indexation des pair_id
for i, p in enumerate(train_data): p['pair_id'] = f"train_{i:04d}"
for i, p in enumerate(test_data):  p['pair_id'] = f"test_{i:04d}"

import collections
print("\n" + "=" * 55)
print("RÉPARTITION FINALE DU DATASET")
print("=" * 55)
for split_name, split in [("TRAIN (300)", train_data), ("TEST  (120)", test_data)]:
    print(f"\n  {split_name}")
    by_ds = collections.Counter(p['dataset_type']   for p in split)
    by_qt = collections.Counter(p['question_type']  for p in split)
    print(f"    Par source     : {dict(by_ds)}")
    print(f"    Par type Q     : {dict(by_qt)}")
print(f"\n  Totaux : {len(train_data)} train + {len(test_data)} test = {len(train_data)+len(test_data)} paires")

## 6. Sauvegarde sur Drive

In [ ]:
PROCESSED_PATH = os.path.join(BASE_PATH, 'data', 'processed')
os.makedirs(PROCESSED_PATH, exist_ok=True)

train_path = os.path.join(PROCESSED_PATH, 'train.json')
test_path  = os.path.join(PROCESSED_PATH, 'test.json')

for data, path, label in [(train_data, train_path, 'train'), (test_data, test_path, 'test')]:
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"{label}.json : {len(data)} paires  ({os.path.getsize(path)/1024:.1f} Ko)  → {path}")

## 7. Statistiques du dataset

In [ ]:
def field_stats(pairs, field):
    counts = [len(str(p.get(field, '')).split()) for p in pairs]
    return min(counts), round(statistics.mean(counts),1), int(statistics.median(counts)), max(counts), sum(counts)

def print_stats(split_name, pairs):
    print(f'\n  ── {split_name} ({len(pairs)} paires) ──')
    print(f'  {"Champ":<12} {"Min":>5} {"Moy":>7} {"Méd":>6} {"Max":>5} {"Total":>10}')
    print(f'  {"─"*12} {"─"*5} {"─"*7} {"─"*6} {"─"*5} {"─"*10}')
    for field, label in [("question","Question"),("answer","Réponse"),("context","Contexte")]:
        mn,mv,md,mx,tot = field_stats(pairs, field)
        print(f'  {label:<12} {mn:>5} {mv:>7} {md:>6} {mx:>5} {tot:>10,}')
    sources = {}
    for p in pairs:
        s = p.get('source','?'); sources[s] = sources.get(s,0)+1
    print(f'  Sources : {dict(sorted(sources.items(), key=lambda x:-x[1]))}')

print('=' * 55)
print('STATISTIQUES DU DATASET (langue : français)')
print('=' * 55)
print_stats('TRAIN', train_data)
print_stats('TEST ', test_data)
all_text = ' '.join(p.get('question','') + ' ' + p.get('answer','') for p in all_qa_pairs)
import re as _re
vocab = set(_re.sub(r'[^a-zàâéèêëîïôùûüç\s]','',all_text.lower()).split())
print(f'\n  Vocabulaire unique : {len(vocab):,} tokens')
print(f'  Total paires       : {len(train_data)+len(test_data)} (train {len(train_data)} + test {len(test_data)})')
print('=' * 55)
print('\n✔ Notebook 02b terminé. Lancez 03_baseline_rag.ipynb pour la suite.')